# C06 · Exceptions, EXC_RETURN and fault dumps

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/01_c_bare_metal/C06_exceptions_exc_return_and_fault_dumps.ipynb)

**Track:** C and bare-metal · **Skill axis (sheet):** C and bare-metal · **Time:** about 60 min

**Skill Hub concepts:** Exception entry and stacking; EXC_RETURN; Fault status registers

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("C06")

## Map: what the core does when an exception fires

Hardware pushes 8 words onto the active stack, then loads the handler address and puts a magic `EXC_RETURN` value in `LR`.

```
 higher addresses
 SP+28  xPSR
 SP+24  PC   <- the instruction that faulted (or the next one, for interrupts)
 SP+20  LR   <- caller's return address at fault time
 SP+16  R12
 SP+12  R3
 SP+8   R2
 SP+4   R1
 SP+0   R0   <- SP after stacking (MSP or PSP)
```

| EXC_RETURN | Return to | Stack holding the frame |
|---|---|---|
| `0xFFFFFFF1` | handler mode | MSP |
| `0xFFFFFFF9` | thread mode | MSP |
| `0xFFFFFFFD` | thread mode | PSP (an RTOS task) |
| `0xFFFFFFE1/E9/ED` | same three, with the floating-point extended frame (Cortex-M4F) | |

Bit 2 of `EXC_RETURN` picks the stack: 0 = MSP, 1 = PSP.

| Register (address) | Field bits | Meaning |
|---|---|---|
| CFSR `0xE000ED28` | [7:0] MemManage | 0 IACCVIOL, 1 DACCVIOL, 3 MUNSTKERR, 4 MSTKERR |
| | [15:8] BusFault | 8 IBUSERR, 9 PRECISERR, 10 IMPRECISERR, 11 UNSTKERR, 12 STKERR |
| | [31:16] UsageFault | 16 UNDEFINSTR, 17 INVSTATE, 18 INVPC, 19 NOCP, 24 UNALIGNED, 25 DIVBYZERO |
| HFSR `0xE000ED2C` | 30 FORCED | a configurable fault escalated to HardFault |
| CCR `0xE000ED14` | 4 DIV_0_TRP | trap integer divide by zero |

## Task 1: decoders (portable C, tested on the host)

In [ ]:
%%writefile decode.c
#include <stdint.h>

const char *stack_used(uint32_t exc_return) {
    (void)exc_return; return "?"; /* TODO */
}

const char *cfsr_cause(uint32_t cfsr) {
    /* name of the LOWEST set bit among the ones in the table, or "NONE" */
    (void)cfsr; return "NONE"; /* TODO */
}

In [ ]:
lab.check("decode")

## Task 2: a HardFault handler that tells you what happened
The naked shim (given) passes the frame pointer and `EXC_RETURN` before the compiler can touch the stack.
You write `fault_report`: print the stacked PC, `EXC_RETURN`, CFSR, the cause and the stack, then exit.
`main.c` enables `DIV_0_TRP` and divides by zero inside `do_div`.

In [ ]:
%%writefile fault.c
#include <stdint.h>
#include "semi.h"
const char *stack_used(uint32_t exc_return);
const char *cfsr_cause(uint32_t cfsr);

__attribute__((naked)) void HardFault_Handler(void) {
    __asm volatile("tst lr, #4      \n"   /* which stack holds the frame? */
                   "ite eq          \n"
                   "mrseq r0, msp   \n"
                   "mrsne r0, psp   \n"
                   "mov r1, lr      \n"   /* EXC_RETURN */
                   "b fault_report  \n");
}

void fault_report(uint32_t *frame, uint32_t exc_return) {
    uint32_t cfsr = *(volatile uint32_t *)0xE000ED28u;
    (void)frame; (void)exc_return; (void)cfsr;
    /* TODO: print pc= (stacked PC), exc_return=, cfsr= with sh_puthex,
             then cause=<name> and stack=<MSP|PSP> lines with sh_puts */
    sh_exit(0);
}

In [ ]:
%%writefile main.c
#include <stdint.h>
#include "semi.h"
volatile int divisor = 0;
__attribute__((noinline)) int do_div(int a, int b) { return a / b; }
int main(void) {
    *(volatile uint32_t *)0xE000ED14u |= (1u << 4);   /* CCR.DIV_0_TRP */
    divisor = do_div(10, divisor);
    sh_puts("no fault\n");
    sh_exit(1);
    return 0;
}

In [ ]:
lab.check("handler")

Cross-check the dump against the disassembly: the stacked PC should point at the `sdiv` instruction.

In [ ]:
from roboprep import clab
print(clab.arm_tool("arm-none-eabi-objdump -d fw.elf | grep -A3 '<do_div>:'"))

## Checkpoint: Math
**M1 (Apply).** After stacking, SP = `0x2000_3FE0`. Address of the stacked PC?

**M2 (Analyze).** CFSR = `0x0200_0000`. Bit number of the set bit?

**M3 (Apply).** Inside the handler, `LR = 0xFFFFFFFD`. Value of bit 2?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Understand).** Why is the first part of the handler a naked assembly shim instead of plain C?
- a) C cannot read special registers at all
- b) a C prologue may push registers and move SP before you capture it, so the frame pointer would be wrong
- c) naked functions run faster

**K2 (Apply).** A field unit returns with `HFSR = 0x4000_0000`, `CFSR = 0x0000_0082`, `MMFAR = 0x0000_0000`, stacked
PC inside `motor_update`. Explain step by step what happened and how you would find the bug.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `c06.h`).
3. In `roboprep/checks/c06.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()